# tm_score-only production pipeline -- z-floor tightening: static vs. dynamic (K = 10, 20, 30)

**Question, updated from the previous version of this notebook.** That version reported
precision@100 and used a single fixed `DEEP_FLOOR_Z = +1.5` for the "tightened" variant. This
version instead:

1. Reports precision **only at K = 10, 20, 30** (no 50, no 100) -- these are the budgets that
  actually matter for this analysis.
2. Adds a **third branch**: instead of one fixed z applied to every ROI (which yields a
  *different* number of surviving candidates per ROI, since raw score distributions differ
  ROI to ROI), this branch solves **per ROI** for the z value that keeps exactly
  `DYNAMIC_TARGET_COUNT = 100` local maxima *before* NMS, then runs NMS and reports
  precision@10/20/30 from there, exactly like the other two branches.

**A mathematical fact this notebook proves in-line, not just states:** the set of pixels that
*can* be a local maximum is decided entirely by `cv2.dilate` (`fused >= dilated`), which does
not depend on the score floor at all. The floor -- whether expressed as a raw score, a z-value,
or "keep the top N by score" (`max_peaks`) -- only decides *how far down* that already-fixed,
already-ordered list of local maxima you cut. Consequently, **"dynamically solve for the z that
keeps N candidates" and "use `max_peaks=N` at any sufficiently permissive floor" are the same
operation and must return the identical candidate set.** This notebook derives the dynamic z
from the baseline branch's own sorted output (an O(1) lookup, effectively free), then verifies
it by running a completely independent, from-scratch `extract_peaks` call at that derived cut
and asserting it reproduces that exact slice, bit for bit -- not approximately. The timing
section reports both numbers separately: the cost of *deriving* the value (expected to be
negligible) and the cost of an independent extraction pass *at* that value (expected to equal
an ordinary stage-4 extraction, since it is one).

`DYNAMIC_TARGET_COUNT = 100` was kept at 100 (not lowered to match the new 10/20/30 reporting
budgets) deliberately: capping the *pre-NMS* pool at exactly the reporting budget was already
measured, in the sibling `max_peaks_100_variant.ipynb` notebook, to under-deliver that budget
after NMS and self-hit removal on every ROI. 100 gives headroom for that attrition while still
being a fixed, ROI-independent target -- the property that motivated this branch in the first
place.

**Everything else is held fixed at production's accepted defaults** (D8_TEMPLATE_ANCHOR.md,
current as of 2026-09-10): `hematoxylin_od` channel, `TM_CCOEFF` (unnormalized), single-scale/
single-angle/no-flip augmentation, `peak_min_distance=7`, `self_hit_radius=5.0`, NMS radius =
match radius = 7.5 um, same 14 ROIs (`images/extra_valid`), same one seed per ROI drawn on the
same RNG stream (`[seed_index, image_id]`), same `tightened_template_box` seed refinement.

**Design choice -- one match per ROI, three extraction branches off it.** None of
`score_threshold`, `max_peaks`, or the dynamic-z derivation affects anything upstream of
`extract_peaks`, so this notebook runs `cv2.matchTemplate` **once** per ROI and reuses that
exact `fused`/`valid` array for all three branches. Any timing or precision difference between
branches is then attributable only to the extraction parameter that changed.

**Sequential execution matters for the timing half of this question.** This notebook must not
be executed concurrently with its sibling ablation notebook.

**Verification built into this notebook:** (1) the baseline branch is checked against the
committed production reference CSV on `seed_ann_id`, `base_size`, `n_detections`, `map_median`,
`mad_scale`, and `tp_at_10`/`tp_at_20`/`tp_at_30` for the `tm_score` arm. (2) every branch's
pool is checked for the seed-annulus invariant and score-descending order post-NMS. (3) the
dynamic-z branch's independently-extracted candidate set is checked bit-for-bit against the
cheap slice of baseline's own sorted output, on every ROI.


In [1]:
import gc
import time
import sys

import cv2
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils import compare as cp
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

NB_T0 = time.time()

# ---------------------------------------------------------------------------------------
# Config -- identical to production_seed_precision_at_k_chromatin_half_pix_fix.ipynb's cell 1
# (D8_TEMPLATE_ANCHOR.md, current production as of 2026-09-10), and to
# latency_profiling/tm_score_latency_profile.ipynb, which reuses that exact config verbatim.
# Nothing here changes except the ablation block below.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0

CHANNEL = 'hematoxylin_od'
METHOD = cv2.TM_CCOEFF
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 5.0

NMS_RADIUS_UM = ev.MIDOG_RADIUS_UM
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=1, flips=(False,), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
BORDER = CFG.patch_size // 2
OTSU_WINDOW = tm.BASE_SIZE

# ---------------------------------------------------------------------------------------
# The three branches.
#
# `extract_peaks`'s `score_threshold` argument is `cut = median + z * MAD`, a per-ROI
# adaptive floor (`template_match.robust_stats`). Production's `DEEP_FLOOR_Z = -1.5` is
# deliberately "near-unfiltered".
#
# Branch 2 ("variant"): one fixed z applied to every ROI, `-1.5 + 3 = +1.5`.
# Branch 3 ("dynamic_z"): a per-ROI z, solved so extraction keeps exactly
#   DYNAMIC_TARGET_COUNT local maxima before NMS on *that* ROI -- see the markdown above for
#   why this is mathematically identical to `max_peaks=DYNAMIC_TARGET_COUNT`.
BASELINE_DEEP_FLOOR_Z = -1.5       # production default
BASELINE_MAX_PEAKS = 2_000_000     # unchanged -- above the theoretical max, never binds

VARIANT_DEEP_FLOOR_Z = 1.5         # -1.5 + 3, static, same for every ROI
VARIANT_MAX_PEAKS = 2_000_000      # unchanged in this notebook
VARIANT_LABEL = 'z=+1.5 (static)'

DYNAMIC_TARGET_COUNT = 100         # pre-NMS local-maxima count the per-ROI z targets
DYNAMIC_LABEL = 'dynamic z (100 pre-NMS candidates/ROI)'

BUDGETS_FOR_EVAL = (10, 20, 30)    # the only budgets this run reports -- no 50, no 100
BUDGET = max(BUDGETS_FOR_EVAL)     # 30 -- stage 6's rank-and-truncate target

REFERENCE_RAW_CSV = '../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv'

print(f'BUDGETS_FOR_EVAL={BUDGETS_FOR_EVAL}, NMS radius = match radius = {NMS_RADIUS_UM} um, '
      f'channel={CHANNEL}')


BUDGETS_FOR_EVAL=(10, 20, 30), NMS radius = match radius = 7.5 um, channel=hematoxylin_od


## Helpers

In [2]:
def draw_seed_with_retry(pool, rng, check_fn):
    """Draw a row via `rng.integers`; on failure drop it and redraw on the same stream."""
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, ref_xy):
    """NMS at `radius`, then drop the template's own self-correlation."""
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - ref_xy[0], c[:, 1] - ref_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def roi_files(images_dir=IMAGES_DIR):
    import os
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')


14 ROIs on disk in ../images/extra_valid/


## Per-ROI worker -- baseline / static z / dynamic z, one match reused three ways

In [3]:
def run_roi(fn, image_id, domain, anns):
    gc.disable()

    # =================== SETUP (once per ROI; not part of click latency) ==============
    t0 = time.perf_counter()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    match_radius = ev.radius_px(mpp, MATCH_RADIUS_UM)
    nms_radius = ev.radius_px(mpp, NMS_RADIUS_UM)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb

    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)
    rng = np.random.default_rng([SEED_INDEX, image_id])

    def _check(row):
        r = ss.tightened_template_box(gray_inv, float(row['cx']), float(row['cy']),
                                      otsu_window=OTSU_WINDOW)
        if r is None:
            return None
        if tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size) is None:
            return None
        return r

    seed, seed_tpl_probe, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    seed_ann_id = int(seed['ann_id'])
    click_cx, click_cy = float(seed['cx']), float(seed['cy'])
    gt_eval = gt[gt['ann_id'] != seed_ann_id].reset_index(drop=True)
    n_gt = int((gt_eval['category_id'] == ds.MITOTIC).sum())
    t_setup = time.perf_counter() - t0

    # ============= PIPELINE stages 1-3 (shared by all branches; timed once) ============
    stages_shared = {}
    t_outer0 = time.perf_counter()

    t1 = time.perf_counter()
    r = ss.tightened_template_box(gray_inv, click_cx, click_cy, otsu_window=OTSU_WINDOW)
    assert r is not None, f'{fn}: seed was pre-validated in SETUP but refused here'
    border_probe = tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size)
    assert border_probe is not None, f'{fn}: seed was pre-validated in SETUP but unreadable here'
    assert r == seed_tpl_probe, f'{fn}: single-shot refinement diverged from the SETUP search'
    base_size, tpl_cx, tpl_cy = r
    tpl_xy = (float(tpl_cx), float(tpl_cy))
    stages_shared['t1_refine_seed_box_s'] = time.perf_counter() - t1

    t2 = time.perf_counter()
    patch = tm.read_padded_patch(hem, *tpl_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    stages_shared['t2_patch_template_build_s'] = time.perf_counter() - t2

    t3 = time.perf_counter()
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    stages_shared['t3_template_matching_s'] = time.perf_counter() - t3

    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    med, mad = tm.robust_stats(fused, valid)

    # ---- one extraction+NMS+precision branch, given an explicit cut value -------------
    def run_branch(branch_name, cut, max_peaks, z_report):
        b = {}
        t4 = time.perf_counter()
        centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, max_peaks)
        b['t4_threshold_peak_extraction_s'] = time.perf_counter() - t4
        n_peaks = len(centers)

        t5 = time.perf_counter()
        c, s = suppress(centers, scores, nms_radius, tpl_xy)
        b['t5_nms_selfhit_s'] = time.perf_counter() - t5
        assert bool(np.all(np.diff(s) <= 0)), \
            f'{fn}/{branch_name}: post-NMS pool is not score-descending'

        t6 = time.perf_counter()
        pool = pd.DataFrame({'cx': c[:, 0], 'cy': c[:, 1], 'score': s})
        top = pool.sort_values('score', ascending=False, na_position='last',
                               kind='mergesort').head(BUDGET)
        b['t6_rank_top_s'] = time.perf_counter() - t6

        d_seed = (np.hypot(pool['cx'] - tpl_xy[0], pool['cy'] - tpl_xy[1])
                 if len(pool) else np.array([]))
        n_near_seed = int((d_seed <= match_radius).sum())

        arm = cp.Arm('tm_score', (lambda d=pool: d), rank_key='score', seeded=True,
                     z=z_report, z_dependent=True, nms_radius=nms_radius,
                     caps=(max_peaks,), coverage_key=None)
        ctx = dict(file_name=fn, tumor_type=domain, image_id=image_id, seed_ann_id=seed_ann_id,
                  base_size=base_size, branch=branch_name, deep_floor_z=z_report,
                  max_peaks=max_peaks, n_retries=n_retries,
                  map_median=round(float(med), 5), mad_scale=round(float(mad), 5))
        checks = []
        ev_out = cp.evaluate_arms([arm], gt_eval, match_radius, roi_shape=(H, W), mpp=mpp,
                                  budgets=BUDGETS_FOR_EVAL, context=ctx, checks=checks)
        checks.append(dict(check='seed_annulus_empty', label=f'{fn}/{branch_name}',
                           n_near_seed=n_near_seed, passed=bool(n_near_seed == 0)))

        return dict(stages=b, n_peaks=n_peaks, n_detections=len(pool), ev_out=ev_out,
                   checks=checks, cut=cut, centers=centers, scores=scores)

    baseline = run_branch('baseline', med + BASELINE_DEEP_FLOOR_Z * mad, BASELINE_MAX_PEAKS,
                          BASELINE_DEEP_FLOOR_Z)
    variant = run_branch('variant', med + VARIANT_DEEP_FLOOR_Z * mad, VARIANT_MAX_PEAKS,
                         VARIANT_DEEP_FLOOR_Z)

    # ---- dynamic z: derive per-ROI from baseline's own sorted output (near-free), then
    # verify with an independent extraction at that exact cut (fair timing + correctness
    # proof) -----------------------------------------------------------------------------
    target = min(DYNAMIC_TARGET_COUNT, len(baseline['scores']))
    assert target == DYNAMIC_TARGET_COUNT, (
        f'{fn}: baseline pool ({len(baseline["scores"])}) is smaller than '
        f'DYNAMIC_TARGET_COUNT ({DYNAMIC_TARGET_COUNT}) -- the near-unfiltered baseline '
        f'floor should never bind here'
    )

    t4a = time.perf_counter()
    cut_dynamic = float(baseline['scores'][target - 1])
    z_dynamic = float((cut_dynamic - med) / mad) if mad > 0 else float('nan')
    cheap_centers = baseline['centers'][:target]
    cheap_scores = baseline['scores'][:target]
    t_z_derivation = time.perf_counter() - t4a

    dynamic = run_branch('dynamic_z', cut_dynamic, 2_000_000, z_dynamic)
    dynamic['stages']['t4a_z_derivation_s'] = t_z_derivation

    # Verification: an independent, from-scratch extraction at the derived cut must
    # reproduce the cheap slice of baseline's own sorted output exactly.
    n_fresh = len(dynamic['centers'])
    n_match = min(n_fresh, target)
    centers_match = bool(np.array_equal(dynamic['centers'][:n_match], cheap_centers[:n_match]))
    scores_match = bool(np.array_equal(dynamic['scores'][:n_match], cheap_scores[:n_match]))
    count_ok = bool(n_fresh >= target)
    dyn_equiv_check = dict(
        check='dynamic_z_reproduces_max_peaks_slice', label=fn,
        n_fresh_extraction=n_fresh, n_target=target,
        passed=bool(centers_match and scores_match and count_ok),
    )

    t_outer_total = time.perf_counter() - t_outer0
    gc.enable()

    del hem, gray_inv, hem_p, fused_p, valid_p, fused, valid, templates, patch
    gc.collect()

    timing_row = dict(
        file_name=fn, tumor_type=domain, base_size=base_size, n_retries=n_retries,
        seed_ann_id=seed_ann_id, map_median=round(float(med), 5), mad_scale=round(float(mad), 5),
        t_setup_s=round(t_setup, 5),
        t1_refine_seed_box_s=round(stages_shared['t1_refine_seed_box_s'], 5),
        t2_patch_template_build_s=round(stages_shared['t2_patch_template_build_s'], 5),
        t3_template_matching_s=round(stages_shared['t3_template_matching_s'], 5),
        n_peaks_baseline=baseline['n_peaks'], n_detections_baseline=baseline['n_detections'],
        t4_baseline_s=round(baseline['stages']['t4_threshold_peak_extraction_s'], 5),
        t5_baseline_s=round(baseline['stages']['t5_nms_selfhit_s'], 5),
        t6_baseline_s=round(baseline['stages']['t6_rank_top_s'], 5),
        n_peaks_variant=variant['n_peaks'], n_detections_variant=variant['n_detections'],
        t4_variant_s=round(variant['stages']['t4_threshold_peak_extraction_s'], 5),
        t5_variant_s=round(variant['stages']['t5_nms_selfhit_s'], 5),
        t6_variant_s=round(variant['stages']['t6_rank_top_s'], 5),
        n_peaks_dynamic=dynamic['n_peaks'], n_detections_dynamic=dynamic['n_detections'],
        z_dynamic=round(z_dynamic, 4),
        t4a_z_derivation_s=round(t_z_derivation, 6),
        t4_dynamic_s=round(dynamic['stages']['t4_threshold_peak_extraction_s'], 5),
        t5_dynamic_s=round(dynamic['stages']['t5_nms_selfhit_s'], 5),
        t6_dynamic_s=round(dynamic['stages']['t6_rank_top_s'], 5),
        t_outer_total_s=round(t_outer_total, 5),
    )

    print(f"[{fn}] {domain:32s} base={base_size:2d} "
          f"n_det base/var/dyn={baseline['n_detections']:6d}/{variant['n_detections']:6d}/"
          f"{dynamic['n_detections']:4d}  n_peaks_dyn={dynamic['n_peaks']:4d} "
          f"z_dyn={z_dynamic:+.3f}  dyn_equiv_check="
          f"{'OK' if dyn_equiv_check['passed'] else 'FAIL'}", flush=True)

    all_checks = baseline['checks'] + variant['checks'] + dynamic['checks'] + [dyn_equiv_check]
    return timing_row, baseline['ev_out'], variant['ev_out'], dynamic['ev_out'], all_checks


## Run -- all 14 ROIs

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

timing_rows, baseline_evs, variant_evs, dynamic_evs, all_checks = [], [], [], [], []
t_run = time.time()
for fn in files:
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    trow, b_ev, v_ev, d_ev, checks = run_roi(fn, image_id, domain, annotations)
    timing_rows.append(trow)
    baseline_evs.append(b_ev)
    variant_evs.append(v_ev)
    dynamic_evs.append(d_ev)
    all_checks.extend(checks)
    gc.collect()

TIMING = pd.DataFrame(timing_rows).set_index('file_name')
BASELINE_RAW = pd.concat(baseline_evs, ignore_index=True)
VARIANT_RAW = pd.concat(variant_evs, ignore_index=True)
DYNAMIC_RAW = pd.concat(dynamic_evs, ignore_index=True)
CHECKS = pd.DataFrame(all_checks)

print(f'\n{len(files)} ROIs timed in {time.time() - t_run:.0f}s')
print(f'checks passed: {int(CHECKS["passed"].sum())}/{len(CHECKS)}')
if not CHECKS['passed'].all():
    display(CHECKS[~CHECKS['passed']])
assert CHECKS['passed'].all(), 'a pipeline-invariant check failed -- see CHECKS above'
CHECKS['check'].value_counts()


[013.tiff] human breast cancer              base=31 n_det base/var/dyn= 17724/ 10198/  97  n_peaks_dyn= 100 z_dyn=+16.156  dyn_equiv_check=OK


[094.tiff] human breast cancer              base=25 n_det base/var/dyn= 18628/ 14420/  96  n_peaks_dyn= 100 z_dyn=+15.303  dyn_equiv_check=OK


[201.tiff] canine lung cancer               base=51 n_det base/var/dyn= 15848/ 10689/  98  n_peaks_dyn= 100 z_dyn=+5.230  dyn_equiv_check=OK


[233.tiff] canine lung cancer               base=25 n_det base/var/dyn= 17449/ 12698/  97  n_peaks_dyn= 100 z_dyn=+8.295  dyn_equiv_check=OK


[245.tiff] canine lymphosarcoma             base=47 n_det base/var/dyn= 17678/ 15635/  89  n_peaks_dyn= 100 z_dyn=+4.623  dyn_equiv_check=OK


...

[460.tiff] canine soft tissue sarcoma       base=47 n_det base/var/dyn= 15698/  9976/  99  n_peaks_dyn= 100 z_dyn=+6.042  dyn_equiv_check=OK


[529.tiff] human melanoma                   base=37 n_det base/var/dyn= 16620/ 10626/  97  n_peaks_dyn= 100 z_dyn=+10.474  dyn_equiv_check=OK


[548.tiff] human melanoma                   base=29 n_det base/var/dyn= 17839/ 13673/  98  n_peaks_dyn= 100 z_dyn=+11.516  dyn_equiv_check=OK



14 ROIs timed in 74s
checks passed: 140/140


check
no_cap                                  42
nms_radius                              42
seed_annulus_empty                      42
dynamic_z_reproduces_max_peaks_slice    14
Name: count, dtype: int64

## Verification -- baseline branch reproduces committed production output

If this fails, nothing below can be trusted.

In [5]:
oracle = pd.read_csv(REFERENCE_RAW_CSV)
oracle_tm = oracle[oracle['arm'] == 'tm_score'].copy()

oracle_roi = (oracle_tm.drop_duplicates('file_name')
             .set_index('file_name')[['seed_ann_id', 'base_size', 'n_detections',
                                        'map_median', 'mad_scale']])
this_roi = TIMING[['seed_ann_id', 'base_size', 'n_detections_baseline', 'map_median',
                   'mad_scale']].rename(columns={'n_detections_baseline': 'n_detections'})

cmp = this_roi.join(oracle_roi, lsuffix='_this', rsuffix='_oracle')
mismatches = []
for col in ['seed_ann_id', 'base_size', 'n_detections']:
    bad = cmp[f'{col}_this'].astype(int) != cmp[f'{col}_oracle'].astype(int)
    if bad.any():
        mismatches.append((col, cmp.index[bad].tolist()))
for col in ['map_median', 'mad_scale']:
    bad = ~np.isclose(cmp[f'{col}_this'], cmp[f'{col}_oracle'], rtol=0, atol=1e-5)
    if bad.any():
        mismatches.append((col, cmp.index[bad].tolist()))

# tp_at_K cross-check for every budget this run actually reports (10, 20, 30): confirms the
# ranking + bucketing + precision computation (not just the search + NMS) reproduces
# production exactly, at every budget this notebook cares about.
oracle_tp = oracle_tm.pivot_table(index='file_name', values='tp_at_budget', columns='budget')
this_baseline_tp = BASELINE_RAW.pivot_table(index='file_name', values='tp_at_budget', columns='budget')
for k in BUDGETS_FOR_EVAL:
    this_k = this_baseline_tp[k]
    oracle_k = oracle_tp.loc[this_k.index, k]
    bad = this_k.astype(int) != oracle_k.astype(int)
    if bad.any():
        mismatches.append((f'tp_at_{k}', this_k.index[bad].tolist()))

if mismatches:
    print('!! MISMATCH vs. committed production reference -- baseline branch diverged:')
    for col, rois in mismatches:
        print(f'   {col}: {rois}')
    display(cmp)
    raise AssertionError('baseline branch does not reproduce the accepted production pipeline')

print(f"All {len(cmp)} ROIs' baseline branch matches {REFERENCE_RAW_CSV} exactly on "
      f"seed_ann_id / base_size / n_detections / map_median / mad_scale / "
      f"tp_at_{{{', '.join(str(k) for k in BUDGETS_FOR_EVAL)}}} (tm_score) -- this notebook's "
      f"harness reproduces production exactly before any variant is applied.")


All 14 ROIs' baseline branch matches ../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv exactly on seed_ann_id / base_size / n_detections / map_median / mad_scale / tp_at_{10, 20, 30} (tm_score) -- this notebook's harness reproduces production exactly before any variant is applied.


## Cross-check -- does dynamic_z actually reproduce a max_peaks=100 candidate set?

This is the in-notebook proof of the equivalence claimed in the introduction: on every ROI, an
independent `extract_peaks` call at the per-ROI dynamic cut must return the identical top-100
local maxima (same positions, same scores, bit for bit) as slicing the top 100 out of
baseline's own sorted output. `dyn_equiv_check` above already asserted this per ROI during the
run (it would have raised if any ROI failed); this cell just reports the aggregate.

In [6]:
equiv_checks = CHECKS[CHECKS['check'] == 'dynamic_z_reproduces_max_peaks_slice']
print(f'{int(equiv_checks["passed"].sum())}/{len(equiv_checks)} ROIs: dynamic_z\'s independent '
      f'extraction exactly reproduces the top-{DYNAMIC_TARGET_COUNT} slice of baseline\'s own '
      f'sorted output.')
equiv_checks[['label', 'n_fresh_extraction', 'n_target', 'passed']]


14/14 ROIs: dynamic_z's independent extraction exactly reproduces the top-100 slice of baseline's own sorted output.


,label,n_fresh_extraction,n_target,passed
9,013.tiff,100.0,100.0,True
19,094.tiff,100.0,100.0,True
29,201.tiff,100.0,100.0,True
39,233.tiff,100.0,100.0,True
49,245.tiff,100.0,100.0,True
59,246.tiff,100.0,100.0,True
69,300.tiff,100.0,100.0,True
79,301.tiff,100.0,100.0,True
89,402.tiff,100.0,100.0,True
99,403.tiff,100.0,100.0,True


## Table 1 -- per-ROI stage timing, three branches (ms)

In [7]:
STAGE_COLS_SHARED = ['t1_refine_seed_box_s', 't2_patch_template_build_s',
                    't3_template_matching_s']
BRANCH_COLS = ['t4_baseline_s', 't5_baseline_s', 't6_baseline_s',
              't4_variant_s', 't5_variant_s', 't6_variant_s',
              't4a_z_derivation_s', 't4_dynamic_s', 't5_dynamic_s', 't6_dynamic_s']
ALL_TIME_COLS = ['t_setup_s'] + STAGE_COLS_SHARED + BRANCH_COLS + ['t_outer_total_s']

TIMING_MS = TIMING.copy()
for c in ALL_TIME_COLS:
    TIMING_MS[c[:-2] + '_ms'] = (TIMING_MS[c] * 1000).round(4 if c == 't4a_z_derivation_s' else 1)

display_cols = ['tumor_type', 'base_size', 'n_detections_baseline', 'n_detections_variant',
               'n_detections_dynamic', 'z_dynamic', 't_setup_ms', 't1_refine_seed_box_ms',
               't2_patch_template_build_ms', 't3_template_matching_ms', 't4_baseline_ms',
               't4_variant_ms', 't4a_z_derivation_ms', 't4_dynamic_ms', 't5_baseline_ms',
               't5_variant_ms', 't5_dynamic_ms', 't6_baseline_ms', 't6_variant_ms',
               't6_dynamic_ms', 't_outer_total_ms']
TIMING_MS.to_csv('z_floor_tightening_timing.csv')
print(f'-> z_floor_tightening_timing.csv')
TIMING_MS[display_cols]


-> z_floor_tightening_timing.csv


,tumor_type,base_size,n_detections_baseline,n_detections_variant,n_detections_dynamic,z_dynamic,t_setup_ms,t1_refine_seed_box_ms,t2_patch_template_build_ms,t3_template_matching_ms,t4_baseline_ms,t4_variant_ms,t4a_z_derivation_ms,t4_dynamic_ms,t5_baseline_ms,t5_variant_ms,t5_dynamic_ms,t6_baseline_ms,t6_variant_ms,t6_dynamic_ms,t_outer_total_ms
file_name,,,,,,,,,,,,,,,,,,,,,
013.tiff,human breast cancer,31,17724,10198,97,16.1558,3900.1,0.9,0.0,1261.2,255.4,242.3,0.003,240.7,254.1,76.8,0.6,1.0,0.5,0.5,2840.7
094.tiff,human breast cancer,25,18628,14420,96,15.3032,2660.0,0.9,0.0,900.6,247.7,238.6,0.004,239.2,295.1,145.3,0.6,0.6,0.8,0.5,2587.7
201.tiff,canine lung cancer,51,15848,10689,98,5.2302,2275.3,1.1,0.0,981.6,214.8,202.3,0.003,199.5,136.0,66.7,0.6,0.6,0.6,0.4,2214.7
233.tiff,canine lung cancer,25,17449,12698,97,8.2950,2126.9,1.1,0.0,771.5,217.0,201.7,0.004,198.7,205.7,97.3,0.6,0.9,0.5,0.5,2121.0
245.tiff,canine lymphosarcoma,47,17678,15635,89,4.6232,2254.6,1.2,0.0,940.5,223.8,209.2,0.003,199.5,266.0,191.9,0.6,0.6,0.6,0.4,2481.5
246.tiff,canine lymphosarcoma,41,18013,14965,96,7.0410,2553.1,1.4,0.1,987.4,225.8,218.3,0.005,206.5,209.1,107.6,0.6,1.1,0.9,0.4,2463.3
300.tiff,canine cutaneous mast cell tumor,45,17940,14173,98,6.6574,2383.7,1.4,0.0,984.3,238.7,200.3,0.006,196.5,241.2,93.7,0.6,1.6,0.6,0.5,2568.2
301.tiff,canine cutaneous mast cell tumor,41,17710,14812,98,5.4709,2257.8,1.0,0.0,897.2,219.7,199.9,0.005,206.0,165.3,103.9,0.6,0.8,0.8,0.6,2322.1
402.tiff,human neuroendocrine tumor,29,17532,14771,98,8.9312,2803.9,0.9,0.0,972.1,250.8,249.9,0.004,235.8,233.9,134.1,0.6,0.6,0.8,0.4,2627.7


In [8]:
print('Shared stages (identical across all three branches by construction -- one '
      'matchTemplate pass reused):')
print(f'  t1 mean={TIMING_MS["t1_refine_seed_box_ms"].mean():.2f}ms  '
      f't2 mean={TIMING_MS["t2_patch_template_build_ms"].mean():.2f}ms  '
      f't3 mean={TIMING_MS["t3_template_matching_ms"].mean():.2f}ms')
print()

stage_pairs = [('t4_baseline_ms', 't4_variant_ms', 'stage 4 -- baseline vs. static z=+1.5'),
              ('t5_baseline_ms', 't5_variant_ms', 'stage 5 -- baseline vs. static z=+1.5'),
              ('t4_baseline_ms', 't4_dynamic_ms', 'stage 4 -- baseline vs. dynamic-z extraction'),
              ('t5_baseline_ms', 't5_dynamic_ms', 'stage 5 -- baseline vs. dynamic-z extraction')]
for base_col, other_col, label in stage_pairs:
    b_mean, o_mean = TIMING_MS[base_col].mean(), TIMING_MS[other_col].mean()
    delta = o_mean - b_mean
    pct = delta / b_mean * 100 if b_mean else float('nan')
    print(f'{label:46s}  {b_mean:7.2f}ms -> {o_mean:7.2f}ms  delta={delta:+7.2f}ms ({pct:+.1f}%)')

print()
z_deriv_mean_us = TIMING_MS['t4a_z_derivation_ms'].mean() * 1000
z_deriv_max_us = TIMING_MS['t4a_z_derivation_ms'].max() * 1000
print(f'Dynamic-z DERIVATION cost (reading the target-th score off baseline\'s already-sorted '
      f'output, an O(1) lookup): mean={z_deriv_mean_us:.1f}us, max={z_deriv_max_us:.1f}us '
      f'across 14 ROIs -- negligible next to any stage above.')
print(f'Dynamic-z INDEPENDENT EXTRACTION cost (re-running extract_peaks from scratch at the '
      f'derived cut, i.e. NOT reusing baseline\'s arrays): mean='
      f'{TIMING_MS["t4_dynamic_ms"].mean():.1f}ms, versus baseline stage 4 mean='
      f'{TIMING_MS["t4_baseline_ms"].mean():.1f}ms -- statistically the same operation, since '
      f'it is one.')

total_base = TIMING_MS[['t1_refine_seed_box_ms', 't2_patch_template_build_ms',
                        't3_template_matching_ms', 't4_baseline_ms', 't5_baseline_ms',
                        't6_baseline_ms']].sum(axis=1)
total_var = TIMING_MS[['t1_refine_seed_box_ms', 't2_patch_template_build_ms',
                       't3_template_matching_ms', 't4_variant_ms', 't5_variant_ms',
                       't6_variant_ms']].sum(axis=1)
total_dyn = TIMING_MS[['t1_refine_seed_box_ms', 't2_patch_template_build_ms',
                       't3_template_matching_ms', 't4a_z_derivation_ms', 't4_dynamic_ms',
                       't5_dynamic_ms', 't6_dynamic_ms']].sum(axis=1)
d_var = total_var.mean() - total_base.mean()
d_dyn = total_dyn.mean() - total_base.mean()
print(f'\nFull pipeline (stages 1-6) mean: baseline={total_base.mean():.1f}ms')
print(f'  static z=+1.5:  {total_var.mean():.1f}ms  (delta={d_var:+.1f}ms, '
      f'{d_var / total_base.mean() * 100:+.1f}%)')
print(f'  dynamic z (independent extraction, i.e. worst case): {total_dyn.mean():.1f}ms  '
      f'(delta={d_dyn:+.1f}ms, {d_dyn / total_base.mean() * 100:+.1f}%)')


Shared stages (identical across all three branches by construction -- one matchTemplate pass reused):
  t1 mean=1.06ms  t2 mean=0.01ms  t3 mean=993.98ms

stage 4 -- baseline vs. static z=+1.5            239.22ms ->  221.63ms  delta= -17.59ms (-7.4%)
stage 5 -- baseline vs. static z=+1.5            202.61ms ->   99.12ms  delta=-103.49ms (-51.1%)
stage 4 -- baseline vs. dynamic-z extraction     239.22ms ->  219.31ms  delta= -19.91ms (-8.3%)
stage 5 -- baseline vs. dynamic-z extraction     202.61ms ->    0.61ms  delta=-202.01ms (-99.7%)

Dynamic-z DERIVATION cost (reading the target-th score off baseline's already-sorted output, an O(1) lookup): mean=4.1us, max=6.0us across 14 ROIs -- negligible next to any stage above.
Dynamic-z INDEPENDENT EXTRACTION cost (re-running extract_peaks from scratch at the derived cut, i.e. NOT reusing baseline's arrays): mean=219.3ms, versus baseline stage 4 mean=239.2ms -- statistically the same operation, since it is one.

Full pipeline (stages 1-6) mean: 

## Table 2 -- precision@{10,20,30}, three branches

Long format: one row per (ROI, branch, budget). `budget_delivered` is reported explicitly --
a branch that changes the pre-NMS pool size can deliver fewer than the requested K after NMS
and self-hit removal have run on it.

In [9]:
BASELINE_RAW['branch'] = 'baseline'
VARIANT_RAW['branch'] = 'variant'
DYNAMIC_RAW['branch'] = 'dynamic_z'
ALL_RAW = pd.concat([BASELINE_RAW, VARIANT_RAW, DYNAMIC_RAW], ignore_index=True)
ALL_RAW['precision_at_budget'] = (ALL_RAW['tp_at_budget']
                                  / ALL_RAW['budget_delivered'].replace(0, np.nan))

PRECISION_LONG = ALL_RAW[['file_name', 'tumor_type', 'branch', 'budget', 'n_detections',
                          'n_gt_mitotic', 'budget_delivered', 'tp_at_budget',
                          'precision_at_budget', 'recall_at_budget']].copy()
PRECISION_LONG = PRECISION_LONG.sort_values(['file_name', 'budget', 'branch']).reset_index(drop=True)
PRECISION_LONG.to_csv('z_floor_tightening_precision.csv', index=False)
print(f'-> z_floor_tightening_precision.csv  ({len(PRECISION_LONG)} rows = 14 ROIs x 3 branches x '
      f'{len(BUDGETS_FOR_EVAL)} budgets)')

for branch in ['baseline', 'variant', 'dynamic_z']:
    sub = PRECISION_LONG[(PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == BUDGET)]
    n_starved = int((sub['budget_delivered'] < BUDGET).sum())
    print(f'{branch:12s}: delivers fewer than BUDGET={BUDGET} candidates on '
          f'{n_starved}/{len(sub)} ROIs at the largest reported budget')

PRECISION_LONG.round(4)


-> z_floor_tightening_precision.csv  (126 rows = 14 ROIs x 3 branches x 3 budgets)
baseline    : delivers fewer than BUDGET=30 candidates on 0/14 ROIs at the largest reported budget
variant     : delivers fewer than BUDGET=30 candidates on 0/14 ROIs at the largest reported budget
dynamic_z   : delivers fewer than BUDGET=30 candidates on 0/14 ROIs at the largest reported budget


,file_name,tumor_type,branch,budget,n_detections,n_gt_mitotic,budget_delivered,tp_at_budget,precision_at_budget,recall_at_budget
0,013.tiff,human breast cancer,baseline,10,17724,17,10,3,0.3000,0.1765
1,013.tiff,human breast cancer,dynamic_z,10,97,17,10,3,0.3000,0.1765
2,013.tiff,human breast cancer,variant,10,10198,17,10,3,0.3000,0.1765
3,013.tiff,human breast cancer,baseline,20,17724,17,20,4,0.2000,0.2353
4,013.tiff,human breast cancer,dynamic_z,20,97,17,20,4,0.2000,0.2353
5,013.tiff,human breast cancer,variant,20,10198,17,20,4,0.2000,0.2353
6,013.tiff,human breast cancer,baseline,30,17724,17,30,6,0.2000,0.3529
7,013.tiff,human breast cancer,dynamic_z,30,97,17,30,6,0.2000,0.3529
8,013.tiff,human breast cancer,variant,30,10198,17,30,6,0.2000,0.3529
9,094.tiff,human breast cancer,baseline,10,18628,81,10,4,0.4000,0.0494


## Table 2b -- precision@K pivoted for readability

In [10]:
PRECISION_PIVOT = PRECISION_LONG.pivot_table(
    index=['tumor_type', 'file_name'], columns=['budget', 'branch'], values='precision_at_budget'
)
PRECISION_PIVOT.round(4)


budget                                           10                         20                         30                  
branch                                     baseline dynamic_z variant baseline dynamic_z variant baseline dynamic_z variant
tumor_type                       file_name                                                                                 
canine cutaneous mast cell tumor 300.tiff       0.7       0.7     0.7     0.70      0.70    0.70   0.6667    0.6667  0.6667
                                 301.tiff       0.9       0.9     0.9     0.90      0.90    0.90   0.8000    0.8000  0.8000
canine lung cancer               201.tiff       0.3       0.3     0.3     0.25      0.25    0.25   0.2000    0.2000  0.2000
                                 233.tiff       0.3       0.3     0.3     0.25      0.25    0.25   0.2667    0.2667  0.2667
canine lymphosarcoma             245.tiff       0.1       0.1     0.1     0.10      0.10    0.10   0.1000    0.1000  0.1000
                                 246.tiff       1.0       1.0     1.0     0.90      0.90    0.90   0.8000    0.8000  0.8000
canine soft tissue sarcoma       459.tiff       0.6       0.6     0.6     0.55      0.55    0.55   0.6333    0.6333  0.6333
                                 460.tiff       0.7       0.7     0.7     0.50      0.50    0.50   0.4000    0.4000  0.4000
human breast cancer              013.tiff       0.3       0.3     0.3     0.20      0.20    0.20   0.2000    0.2000  0.2000
                                 094.tiff       0.4       0.4     0.4     0.50      0.50    0.50   0.5333    0.5333  0.5333
human melanoma                   529.tiff       0.3       0.3     0.3     0.20      0.20    0.20   0.1333    0.1333  0.1333
                                 548.tiff       0.5       0.5     0.5     0.50      0.50    0.50   0.5333    0.5333  0.5333
human neuroendocrine tumor       402.tiff       0.4       0.4     0.4     0.45      0.45    0.45   0.4667    0.4667  0.4667
                                 403.tiff       0.5       0.5     0.5     0.35      0.35    0.35   0.2667    0.2667  0.2667

## Summary -- pooled precision and win counts, by budget

In [11]:
print(f'Pooled precision (sum tp / sum delivered) across all 14 ROIs, by branch and budget:')
for k in BUDGETS_FOR_EVAL:
    print(f'  K={k}:')
    for branch in ['baseline', 'variant', 'dynamic_z']:
        sub = PRECISION_LONG[(PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == k)]
        pooled = sub['tp_at_budget'].sum() / sub['budget_delivered'].sum()
        print(f'    {branch:12s}: {pooled:.4f}  ({int(sub["tp_at_budget"].sum())} tp / '
              f'{int(sub["budget_delivered"].sum())} delivered)')

print()
for k in BUDGETS_FOR_EVAL:
    base_k = PRECISION_LONG[(PRECISION_LONG['branch'] == 'baseline') & (PRECISION_LONG['budget'] == k)].set_index('file_name')
    var_k = PRECISION_LONG[(PRECISION_LONG['branch'] == 'variant') & (PRECISION_LONG['budget'] == k)].set_index('file_name')
    dyn_k = PRECISION_LONG[(PRECISION_LONG['branch'] == 'dynamic_z') & (PRECISION_LONG['budget'] == k)].set_index('file_name')
    prec_delta_var = (var_k['precision_at_budget'] - base_k['precision_at_budget'])
    prec_delta_dyn = (dyn_k['precision_at_budget'] - base_k['precision_at_budget'])
    recall_down_var = (var_k['recall_at_budget'] < base_k['recall_at_budget']).sum()
    recall_down_dyn = (dyn_k['recall_at_budget'] < base_k['recall_at_budget']).sum()
    print(f'K={k}: static z=+1.5 precision delta: {int((prec_delta_var > 0).sum())} up / '
          f'{int((prec_delta_var < 0).sum())} down / {int((prec_delta_var == 0).sum())} '
          f'unchanged of 14 ROIs; recall lower on {int(recall_down_var)}/14')
    print(f'      dynamic_z    precision delta: {int((prec_delta_dyn > 0).sum())} up / '
          f'{int((prec_delta_dyn < 0).sum())} down / {int((prec_delta_dyn == 0).sum())} '
          f'unchanged of 14 ROIs; recall lower on {int(recall_down_dyn)}/14')


Pooled precision (sum tp / sum delivered) across all 14 ROIs, by branch and budget:
  K=10:
    baseline    : 0.5000  (70 tp / 140 delivered)
    variant     : 0.5000  (70 tp / 140 delivered)
    dynamic_z   : 0.5000  (70 tp / 140 delivered)
  K=20:
    baseline    : 0.4536  (127 tp / 280 delivered)
    variant     : 0.4536  (127 tp / 280 delivered)
    dynamic_z   : 0.4536  (127 tp / 280 delivered)
  K=30:
    baseline    : 0.4286  (180 tp / 420 delivered)
    variant     : 0.4286  (180 tp / 420 delivered)
    dynamic_z   : 0.4286  (180 tp / 420 delivered)

K=10: static z=+1.5 precision delta: 0 up / 0 down / 14 unchanged of 14 ROIs; recall lower on 0/14
      dynamic_z    precision delta: 0 up / 0 down / 14 unchanged of 14 ROIs; recall lower on 0/14
K=20: static z=+1.5 precision delta: 0 up / 0 down / 14 unchanged of 14 ROIs; recall lower on 0/14
      dynamic_z    precision delta: 0 up / 0 down / 14 unchanged of 14 ROIs; recall lower on 0/14
K=30: static z=+1.5 precision delta: 0 up

## Closing readout

In [12]:
print(f'Precision, pooled across 14 ROIs (baseline -> static z=+1.5 -> dynamic_z):')
for k in BUDGETS_FOR_EVAL:
    vals = {}
    for branch in ['baseline', 'variant', 'dynamic_z']:
        sub = PRECISION_LONG[(PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == k)]
        vals[branch] = sub['tp_at_budget'].sum() / sub['budget_delivered'].sum()
    print(f'  K={k}: {vals["baseline"]:.4f} -> {vals["variant"]:.4f} -> {vals["dynamic_z"]:.4f}')

print()
print(f'Timing, full pipeline mean over 14 ROIs:')
print(f'  baseline:                                  {total_base.mean():7.1f}ms')
print(f'  static z=+1.5:                              {total_var.mean():7.1f}ms  '
      f'({d_var:+.1f}ms, {d_var / total_base.mean() * 100:+.1f}%)')
print(f'  dynamic_z (independent extraction, worst case): {total_dyn.mean():7.1f}ms  '
      f'({d_dyn:+.1f}ms, {d_dyn / total_base.mean() * 100:+.1f}%)')
print(f'  dynamic_z z-derivation alone (best case, reusing baseline\'s arrays): '
      f'+{z_deriv_mean_us:.1f}us -- effectively free')
print()
print('Dynamic-z reproduces max_peaks=100 exactly (see the cross-check above), so its '
      'precision/recall numbers are not independent evidence -- they are the same candidate '
      'set already characterised (as max_peaks_100_variant.ipynb) restated in z units. It '
      'does not solve the "variable candidate count" property of a fixed z differently than '
      'max_peaks already does; it reaches the identical fixed-count pool by a different, and '
      'not cheaper, route -- deriving the value is free only when another branch has already '
      'paid for a full sorted extraction, and an independent derivation costs one ordinary '
      'stage-4 extraction pass, no more, no less.')
print()
print(f'Caveat (D5, restated): this is single-seed, n=14 -- one click per ROI, not the paired, '
      f'multi-seed sweep D5 itself sets as the bar for changing a production default.')
print(f'\nnotebook ran in {time.time() - NB_T0:.0f}s')


Precision, pooled across 14 ROIs (baseline -> static z=+1.5 -> dynamic_z):
  K=10: 0.5000 -> 0.5000 -> 0.5000
  K=20: 0.4536 -> 0.4536 -> 0.4536
  K=30: 0.4286 -> 0.4286 -> 0.4286

Timing, full pipeline mean over 14 ROIs:
  baseline:                                   1437.8ms
  static z=+1.5:                               1316.5ms  (-121.3ms, -8.4%)
  dynamic_z (independent extraction, worst case):  1215.4ms  (-222.4ms, -15.5%)
  dynamic_z z-derivation alone (best case, reusing baseline's arrays): +4.1us -- effectively free

Dynamic-z reproduces max_peaks=100 exactly (see the cross-check above), so its precision/recall numbers are not independent evidence -- they are the same candidate set already characterised (as max_peaks_100_variant.ipynb) restated in z units. It does not solve the "variable candidate count" property of a fixed z differently than max_peaks already does; it reaches the identical fixed-count pool by a different, and not cheaper, route -- deriving the value is free on